<a href="https://colab.research.google.com/github/lcbjrrr/DBMS/blob/main/Lab_ACID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Activity: SQL Transactions

A transaction is the mechanism that groups related statements into a single unit of work, so that the database ends up with either all the changes or none, never half. In this activity, you will first create the inconsistency deliberately and see it sitting in the table, then discover that simply wrapping the statements in BEGIN does not rescue you, and finally learn where the real protection comes from: the choice between COMMIT and ROLLBACK after you have seen whether the work actually succeeded.

## Step 1 — Try This Example


Run these two statements one after the other, with no transaction:


In [1]:
import sqlite3
db_path = '/content/ok.sqlite'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # Execute the first INSERT statement
    cursor.execute("INSERT INTO Publishers (publisher_name, impact_factor) VALUES ('SHOULD NOT INSERT', 3.10);")
    # Execute the second INSERT statement
    cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume, number) VALUES (last_insert_rowid(), NULL, 2024, 33, 2);")
    conn.commit()
    print("SQL statements executed successfully.")
except sqlite3.Error as e:
    print(f"An error occurred: {e}")
finally:
    if conn:
        conn.close()

An error occurred: NOT NULL constraint failed: Journals.issn


In [4]:
!sudo apt install sqlite

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libsqlite0 sqlite3
Suggested packages:
  sqlite-doc sqlite3-doc
The following NEW packages will be installed:
  libsqlite0 sqlite sqlite3
0 upgraded, 3 newly installed, 0 to remove and 24 not upgraded.
Need to get 945 kB of archives.
After this operation, 2,331 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libsqlite0 amd64 2.8.17-15fakesync1build1 [160 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 sqlite amd64 2.8.17-15fakesync1build1 [16.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 sqlite3 amd64 3.37.2-2ubuntu0.7 [769 kB]
Fetched 945 kB in 1s (1,801 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dia

In [6]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers;"

1|Nature Publishing|42.77
2|IEEE|10.5
3|ACM|8.2
4|Springer|5.4
5|Elsevier|6.1
6|NeurIPS Foundation|15
7|O'Reilly Media|2.1
8|VLDB Journal|3.1
9|ACM TODsS|2.75
10|xxxx|3.1
11|yyyy|3.1


## Step 2 — Now Talk About Transactions



Same work, but wrapped this time. Write down your prediction before you run it.


In [10]:
import sqlite3
db_path = '/content/ok.sqlite'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO Publishers (publisher_name, impact_factor) VALUES ('NOW IT IS OK', 3.10);")
    cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume, number) VALUES (last_insert_rowid(), '000', 2024, 33, 2);")
    conn.commit()
    print("SQL statements executed successfully within a transaction.")
except sqlite3.Error as e:
    if conn:
        conn.rollback()
    print(f"An error occurred: {e}. Transaction rolled back.")
finally:
    if conn:
        conn.close()

SQL statements executed successfully within a transaction.


In [11]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers;"

1|Nature Publishing|42.77
2|IEEE|10.5
3|ACM|8.2
4|Springer|5.4
5|Elsevier|6.1
6|NeurIPS Foundation|15
7|O'Reilly Media|2.1
8|NOW IT IS OK|3.1
9|NOW IT IS OK|3.1
10|NOW IT IS OK|3.1


In [23]:
import sqlite3
import csv
db_path = '/content/ok.sqlite'
pubs_csv_path = '/content/pubs.csv'
journals_csv_path = '/content/journals.csv'
conn = None
try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    with open(pubs_csv_path, 'r', newline='', encoding='utf-8') as inst_file:
        inst_reader = csv.reader(inst_file)
        for row in inst_reader:
            if len(row) >= 2:
                id = row[0]
                cursor.execute("INSERT INTO Publishers (publisher_id, publisher_name, impact_factor) VALUES (?, ?, ?);", (id,row[1], row[2]))

    with open(journals_csv_path, 'r', newline='', encoding='utf-8') as authors_file:
        authors_reader = csv.reader(authors_file)
        for row in authors_reader:
            if len(row) >= 2:
                  cursor.execute("INSERT INTO Journals (publisher_id, issn, year, volume) VALUES (?,?,?,?);", (id, row[1], row[2], row[3]))
    conn.commit()
    print("Data from Publishers and Journals imported successfully within a single transaction.")

except Exception as e:
    if conn:
        conn.rollback()
    print(f"An unexpected error occurred: {e}. Transaction rolled back.")
finally:
    if conn:
        conn.close()

An unexpected error occurred: [Errno 2] No such file or directory: '/content/journals.csv'. Transaction rolled back.


In [24]:
! sqlite3 ok.sqlite "SELECT * FROM Publishers p, Journals j WHERE p.publisher_id = j.publisher_id;"

1|Nature Publishing|42.77|1|1476-4687|2024|625|7995
2|IEEE|10.5|2|0018-9219|2023|111|5
3|ACM|8.2|3|0001-0782|2024|67|2
4|Springer|5.4|4|0028-0836|2022|50|12
5|Elsevier|6.1|5|0022-2836|2023|435|10
